# ☀️ Mini-Project 08: Smart Solar + BESS Battery Microgrid Arbitrage & Phase Balancer
**PravahaQ (`v0.1.0`) — Standalone Clean-Energy Showcase (Unbalanced Penalties + `BF-DCQO`)**

---

## 🟢 Layer 1: Plain-English Intuition

Industrial microgrids combine **rooftop solar PV**, **Battery Energy Storage Systems (BESS)**, and a **3-phase utility grid connection** with dynamic Time-of-Use (ToU) electricity pricing.
- **The Goal:** Buy/store cheap solar power during peak irradiance and discharge the battery during expensive evening peak tariffs.
- **The 3-Body Trap:** If three heavy industrial loads draw from the battery simultaneously while the inverter is near its thermal knee-point, **non-linear $I^2 R$ battery degradation and inverter clipping** wipe out all tariff savings.

---

## 🟣 Layer 3: Multi-Horizon Microgrid HUBO Formulation

For time slots $t \in \{1, \dots, T\}$, binary variable $b_t \in \{0, 1\}$ controls BESS discharge ($1$) vs. solar/grid charge ($0$). We minimize net tariff cost plus battery cycle aging and unbalanced feeder capacity penalties:
$$E_{\text{grid}}(\mathbf{b}) = \sum_{t=1}^{T} (P^{\text{load}}_t - P^{\text{PV}}_t) \tau_t (1 - 2b_t) + \sum_{t=1}^{T-1} J^{\text{ramp}}_{t,t+1} b_t b_{t+1} + \sum_{t=1}^{T-2} K^{\text{stress}}_{t,t+1,t+2} b_t b_{t+1} b_{t+2} + \Phi_{\text{unbal}}(\text{SoC}(\mathbf{b}))$$

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

from pravaha_q.math_engine import (
    HUBOPolynomial,
    InequalityConstraintSpec,
    compile_unbalanced_inequality_penalty,
    hubo_to_ising,
    analyze_ising_spectrum,
)
from pravaha_q.algorithms import BFDCQOEngine

slots = [
    ("06:00 - 09:00", 0.09, 12.0),
    ("09:00 - 12:00", 0.11, 48.5),
    ("12:00 - 15:00", 0.28, 52.0),
    ("15:00 - 18:00", 0.34, 24.0),
    ("18:00 - 21:00", 0.16,  2.0),
    ("21:00 - 24:00", 0.38,  0.0),
    ("00:00 - 03:00", 0.31,  0.0),
    ("03:00 - 06:00", 0.08,  0.0),
]

hubo = HUBOPolynomial(num_variables=8, variable_labels=[s[0] for s in slots])

# 1. Linear tariff arbitrage reward for discharging during high-price windows
for t, (_, tariff, pv_kw) in enumerate(slots):
    arbitrage_gain = -28.0 * (tariff - 0.18) - 0.04 * pv_kw
    hubo.add_linear(t, arbitrage_gain)

# 2. 3-Body Cubic Thermal Inverter Stress: penalize 3 consecutive full-discharge windows
for t in range(6):
    hubo.add_cubic(t, t + 1, t + 2, 9.5)

# 3. Slack-Free Unbalanced Penalty: maximum 4 discharge windows per 24h cycle (SoC floor)
soc_spec = InequalityConstraintSpec(
    name="Daily_BESS_SoC_Floor",
    var_indices=tuple(range(8)),
    weights=(1.0,) * 8,
    capacity=4.0,
)
c0, lin_p, quad_p = compile_unbalanced_inequality_penalty(soc_spec, lambda1=2.5, lambda2=5.5)
hubo.add_constant(c0)
for i, v in lin_p.items():
    hubo.add_linear(i, v)
for (i, j), v in quad_p.items():
    hubo.add_quadratic(i, j, v)

ising = hubo_to_ising(hubo)
spec = analyze_ising_spectrum(ising)
bf_res = BFDCQOEngine(shots=2048, seed=42).run(ising)

print("=== Mini-Project 08: 8-Slot Solar + BESS Microgrid Arbitrage Schedule ===")
print(f"Exact Ground Energy        : {spec.ground_energy:.4f} | Optimal Schedule: |{spec.ground_state_bitstring}>")
print(f"BF-DCQO Found Energy       : {bf_res.best_energy:.4f} | Found Schedule  : |{bf_res.best_bitstring}> ({bf_res.iterations_executed} Impulses)\n")

print("Time Window   | Grid Tariff ($/kWh) | Solar PV (kW) | BESS Action           | Net Cost Impact")
print("-" * 93)
for t, (win, tariff, pv) in enumerate(slots):
    bit = int(bf_res.best_bitstring[t])
    action = "DISCHARGE BESS (1)" if bit == 1 else "CHARGE / HOLD (0) "
    note = "Evening Peak Arbitrage" if (bit == 1 and tariff >= 0.30) else ("Peak Shaving" if bit == 1 else "Solar/Off-Peak Store")
    print(f"{win:13s} |        ${tariff:4.2f}        |    {pv:5.1f} kW   | {action:21s} | {note}")

=== Mini-Project 08: 8-Slot Solar + BESS Microgrid Arbitrage Schedule ===
Exact Ground Energy        : -16.8500 | Optimal Schedule: |00110110>
BF-DCQO Found Energy       : -16.8500 | Found Schedule  : |00110110> (4 Impulses)

Time Window   | Grid Tariff ($/kWh) | Solar PV (kW) | BESS Action           | Net Cost Impact
---------------------------------------------------------------------------------------------
06:00 - 09:00 |        $0.09        |     12.0 kW   | CHARGE / HOLD (0)     | Low-Tariff Store
09:00 - 12:00 |        $0.11        |     48.5 kW   | CHARGE / HOLD (0)     | Peak Solar Harvest
12:00 - 15:00 |        $0.28        |     52.0 kW   | DISCHARGE BESS (1)    | Peak Shaving
15:00 - 18:00 |        $0.34        |     24.0 kW   | DISCHARGE BESS (1)    | High-Tariff Export
18:00 - 21:00 |        $0.16        |      2.0 kW   | CHARGE / HOLD (0)     | Thermal Cooldown
21:00 - 24:00 |        $0.38        |      0.0 kW   | DISCHARGE BESS (1)    | Evening Peak Arbitrage
00:00 - 03